## **基礎迴圈** (100 min)
### **重複執行的意思**

### **基礎迴圈有兩種**
*   while 迴圈 : 條件判斷為主要控制
*   for 迴圈 : 計數迴圈

### **while 迴圈需要注意的事情：**


*   判斷是否重複執行完全依靠條件決定，如果邏輯設計有誤，則可能發生無限迴圈的情況
*   可以使用 break, continue 關鍵字中斷迴圈的執行



---


### **作業上繳設定**

1. **檔案 → 在雲端硬碟中儲存副本**
2. 對老師分享的 `2026檢定培訓營` 按 **「新增雲端硬碟捷徑」** 加到「我的雲端硬碟」（課堂會引導）
3. 確認老師給的是 **編輯者** 權限（否則無法上傳）
4. 填好下方 **DAY**（Day1、Day2…）與 **STUDENT_NAME**（你的姓名）
5. 依序執行 3 個灰色程式格：設定 → 掛載 Drive → **上繳工具**（應顯示 `上繳工具已載入（2026-06-15）`）
6. 每題流程：**寫程式 → 執行作答格 → 執行 `make_submit_button("APCS_5-1")` 格 → 按上繳**

上繳後檔案會出現在：

```
2026檢定培訓營 / Day1 / 王小明 / APCS_5-1.py
```

> `Day1`、姓名等子資料夾**不用事先建立**，第一次上繳會自動產生。
> 若改過答案，要**再執行一次** `make_submit_button(...)` 那格，再按上繳。


In [ ]:
# 老師分享的共用資料夾（捷徑到「我的雲端硬碟」後的路徑，通常不用改）
SUBMIT_BASE = "/content/drive/MyDrive/橘子蘋果/APCS/2026檢定培訓營"

# ===== 請填入 =====
DAY = "Day1"           # 統一格式：Day1、Day2、Day3...
STUDENT_NAME = "王小明"  # 你的姓名


In [ ]:
#@title 首次使用需授權 Google 帳號
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive 已掛載")


In [ ]:
#@title 上傳相關設定
import os

import ipywidgets as widgets
from IPython.display import clear_output, display
from IPython import get_ipython

_NOTEBOOK_CACHE = None
SUBMIT_TOOL_VERSION = "2026-06-15"


def _user_ns():
    ip = get_ipython()
    return ip.user_ns if ip is not None else globals()


def _resolve_submit_folder():
    ns = _user_ns()
    base = ns.get("SUBMIT_BASE", "")
    day = ns.get("DAY", "").strip()
    name = ns.get("STUDENT_NAME", "").strip()

    if not base:
        raise ValueError("SUBMIT_BASE 未設定")
    if not os.path.isdir(base):
        raise ValueError(
            f"找不到共用資料夾：{base}\n"
            "請確認已掛載 Drive，且已將「2026檢定培訓營」捷徑加到「我的雲端硬碟」"
        )
    if not day:
        raise ValueError("請填 DAY（例如 Day1）")
    if not day.startswith("Day") or not day[3:].isdigit():
        raise ValueError("DAY 請用 Day1、Day2 格式")
    if not name:
        raise ValueError("請填 STUDENT_NAME（你的姓名）")

    return os.path.join(base, day, name)


def _submit_filename(question_id):
    return f"{question_id}.py"


def _cell_source_text(cell):
    src = cell.get("source", [])
    if isinstance(src, str):
        return src
    return "".join(src)


def _capture_notebook_now():
    """在執行 cell 當下讀取 Colab 畫面上的講義（不能在按鈕 callback 裡讀）。"""
    try:
        from google.colab import _message

        reply = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
        if reply and "ipynb" in reply:
            return reply["ipynb"]
    except Exception:
        pass
    return None


def _extract_answer_code(nb, question_id):
    marker = f"#start-{question_id}"
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = _cell_source_text(cell)
        if marker not in src:
            continue
        lines = []
        for line in src.splitlines():
            stripped = line.strip()
            if stripped == marker or stripped == "#start":
                continue
            lines.append(line)
        code = "\n".join(lines).strip()
        return code + "\n" if code else ""
    raise ValueError(
        f"找不到作答區 {marker}。請確認開啟的是「_上繳」版講義，並在作答格寫好程式。"
    )


def make_submit_button(question_id):
    global _NOTEBOOK_CACHE

    nb = _capture_notebook_now()
    if nb is None:
        print("⚠️ 無法讀取講義。請在 Google Colab 中執行此格。")
        _NOTEBOOK_CACHE = None
    else:
        _NOTEBOOK_CACHE = nb
        try:
            _extract_answer_code(nb, question_id)
        except ValueError as exc:
            print(f"⚠️ {exc}")

    btn = widgets.Button(description="上繳", button_style="success", icon="cloud-upload")
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output(wait=True)
            print("上繳中，請稍候…")
            try:
                if _NOTEBOOK_CACHE is None:
                    print("⚠️ 請先重新執行此格（make_submit_button），再按上繳。")
                    return

                if not os.path.isdir("/content/drive/MyDrive"):
                    print("⚠️ 請先執行上方的「掛載 Google Drive」cell")
                    return

                code = _extract_answer_code(_NOTEBOOK_CACHE, question_id)
                if not code.strip():
                    print("⚠️ 作答區是空的，請先寫程式，再重新執行此格，然後按上繳")
                    return

                target_dir = _resolve_submit_folder()
                os.makedirs(target_dir, exist_ok=True)

                filename = _submit_filename(question_id)
                filepath = os.path.join(target_dir, filename)

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(code)

                print("✅ 上繳成功！")
                print(f"檔案：{filename}")
                print(f"路徑：{filepath}")
            except Exception as exc:
                print(f"❌ 上繳失敗：{exc}")

    btn.on_click(on_submit)
    display(widgets.VBox([btn, output]))


print(f"上繳工具已載入（{SUBMIT_TOOL_VERSION}）")

<br>

### **APCS 5-1 迴圈的行為**
#### **題目說明：**
#### 練習使用 while 迴圈，連續輸出 1~5 的數字
``` python
while 需要滿足的條件:
  # 回圈內要執行的程式
  #...
  #...
```

#### **解題想法**
*   while 迴圈是一個需要條件的迴圈，在這個題目中，什麼時候需要離開結束迴圈呢？
*   輸出數字 1 ~ 5 代表程式一開始會有一個數值為 1 ，接著依序累加到 5



In [ ]:
#start
num = 1
while num<=5 :
    print(num)

#### **緊急!**
*   發生"無限迴圈"!!!
*   程式停不下來的時候可以按下鍵盤的 "ctrl+c"

In [ ]:
#start-APCS_5-1


In [ ]:
make_submit_button("APCS_5-1")


<br>

### **APCS 5-2 輸出名條 (自我挑戰)**
#### **題目說明：**
#### 新的學期開始了，為了製作名條方便，老師決定寫個程式自動產生
#### 只要輸入班級人數 N，就會輸出 No.1~No.N
*   此題指定使用 while 迴圈實作
*   1 <= N <= 50

#### **解題想法**
*   使用 while 迴圈需要小心無限迴圈的狀況
*   輸出結果需要使用到字串與變數合併，要注意數值的型態轉換



In [ ]:
#start-APCS_5-2


In [ ]:
make_submit_button("APCS_5-2")


<br>

### **APCS 5-3 range()**
```python
range(5) = 0, 1, 2, 3, 4 #產生 0 ~ N-1 的數列
range(1, 6) = 1, 2, 3, 4, 5 #產生 1 ~ N-1 的數列
range(1, 10, 2) = 1, 3, 5, 7, 9 #產生 1 ~ N-1 的數列, 但每個數字中會間隔2個值
```

### **疑問**: 產生的數列是串列型別嗎?
```python
print(type(range(3)))
<class 'range'>
```

### **印出數列**
```python
for i in range(5):
  print(i)
```

#### **題目說明：**
#### 請使用 for 迴圈輸出以下結果，格式如有換行或空格必須完全相同
```python
1 2 3 4 5 6 7 8 9 10
10 9 8 7 6 5 4 3 2 1
```



#### **解題想法**
*   程式輸出有兩行，需要產生兩個 for 迴圈
*   如何使用 range() 製作由大到小的遞減數列？
*   如果直接 print(i) 則會有換行，如何輸出連續數列不換行？

In [ ]:
#start-APCS_5-3


In [ ]:
make_submit_button("APCS_5-3")


<br>

### **APCS 5-4 找出因數**
#### **題目說明：**
#### 輸入一個正整數 N，列出此數字的所有因數

#### **範例輸入：**
*   10
*   21
*   48

#### **應得輸出：**
*   1 2 5 10
*   1 3 7 21
*   1 2 3 4 6 8 12 16 24 48


#### **解題想法**
#### 此題需要注意的重點
*   目前學會兩種迴圈的語法，依照此題的狀況，你會選擇哪種？
    1. 無法預測或估算迴圈次數，使用 while 迴圈 <br>
    2. 能估算或具有特定規律性，使用 for 迴圈 <br>
*   數學中，如何判斷因數？
*   此題不需要寫成連續 IPO 結構

In [ ]:
#start-APCS_5-4


In [ ]:
make_submit_button("APCS_5-4")


<br>

### **APCS 5-5 數列求和 (自我挑戰)**
#### **題目說明：**
#### 輸入一個正整數 N，請輸出 1+2+3+ … +N 的總和





#### **解題想法：**
*   由題目判斷使用 while or for 迴圈
*   求總和必須增加一個變數，用來儲存目前累加的結果

In [ ]:
#start-APCS_5-5


In [ ]:
make_submit_button("APCS_5-5")


<br>

### **APCS 5-6 數列階乘 (自我挑戰)**
#### **題目說明：**
#### 數學中有一個特殊的用法稱為階乘 (factorial)
#### 輸入一個正整數 N，輸出 1x2x3x … xN 的結果


#### **範例輸入：**
*   4
*   12

#### **應得輸出：**
*   24
*   479001600



#### **解題想法：**
*   由題目判斷使用 while or for 迴圈
*   求階乘的結果需要不斷的連續相乘數字



In [ ]:
#start-APCS_5-6


In [ ]:
make_submit_button("APCS_5-6")


<br>

### **APCS 5-7 99乘法表**
#### **題目說明：**
#### 請利用 for 迴圈建立一個 99 乘法表，輸出請參考以下結果
```python
1x1= 1 1x2= 2 1x3= 3 ...
2x1= 2 2x2= 4 2x3= 6 ...
3x1= 3 3x2= 6 3x3= 9 ...
...
...
9x1= 9 9x2= 18 9x3= 27 ...
```

#### **解題想法：**
*   只用一次 for 迴圈好像沒辦法建立 1x1 ~ 9x9
*   輸出不是只有乘法結果，連算式都需要輸出

In [ ]:
#start-APCS_5-7


In [ ]:
make_submit_button("APCS_5-7")


<br>

### **APCS 5-8a 文字圖形產生器**
#### **題目說明：**
#### 用迴圈與條件式用在排版輸出的應用
#### 透過輸入 M, N 兩個值，可以控制長方形的形狀
```python
*****
*****
*****
*****
```

In [ ]:
#start-APCS_5-8a


In [ ]:
make_submit_button("APCS_5-8a")


<br>

### **APCS 5-8b 文字圖形產生器**
#### **題目說明：**
#### 用迴圈與條件式用在排版輸出的應用
#### 三角形的基本寫法需要使用到兩個 for 迴圈，才能輸出形狀
```python
*
**
***
****
*****
```

In [ ]:
#start-APCS_5-8b


In [ ]:
make_submit_button("APCS_5-8b")


<br>

### **APCS 5-8c 文字圖形產生器**
#### **題目說明：**
#### 畫另外一個方向的三角形
#### 可以把輸出文字當作同時輸出空白和 * 符號
#### 再依照迴圈 i 值的變化，調整輸出的內容
```python
    *
   **
  ***
 ****
*****
```


In [ ]:
#start-APCS_5-8c


In [ ]:
make_submit_button("APCS_5-8c")


<br>

### **APCS 5-9 文字圖形產生器(進階)**
#### **題目說明：**
#### 請使用迴圈加條件式產生以下文字圖形

#### APCS 5-9a.py
```
    *
   * *
  * * *
 * * * *
* * * * *
 * * * *
  * * *
   * *
    *
```

In [ ]:
#start-APCS_5-9a


In [ ]:
make_submit_button("APCS_5-9a")


<br>

#### APCS 5-9b.py
```
0
10
010
1010
01010
```

In [ ]:
#start-APCS_5-9b


In [ ]:
make_submit_button("APCS_5-9b")


<br>

#### APCS 5-9c.py
```
01111
20222
33033
44404
55550
```

In [ ]:
#start-APCS_5-9c


In [ ]:
make_submit_button("APCS_5-9c")


<br>

## 重點複習

*   迴圈是條件式以外的另外一種流程控制
*   迴圈有兩種類型 while 迴圈，for 迴圈
*   while 迴圈主要用在不知道會執行幾次的程式
*   for 迴圈通常會搭配 range() 函數，執行固定的重複次數
*   迴圈除了能重複計算以外，也能搭配條件式使用
*   雙層迴圈是進階的用法，可以重複多層次的迴圈結構

